In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import sys
import time
import seaborn as sns
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from torch.utils.data import DataLoader, Dataset
from collections import Counter
from PIL import Image
from torchvision.transforms import ToPILImage

# Set some options for printing all the columns
np.set_printoptions(precision = 10, threshold = sys.maxsize)
np.set_printoptions(linewidth = np.inf)

pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

pd.options.display.float_format = '{:,.10f}'.format

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [ ]:
from datasets import load_dataset

ds = load_dataset("ethz/food101")

In [ ]:
train_ds = ds['train']
val_ds = ds['validation']

classes = train_ds.features['label'].names

In [ ]:
transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB') if img.mode == 'L' else img),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [ ]:
from torch.utils.data import DataLoader, Dataset

class FoodDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image = self.dataset[idx]['image']
        label = self.dataset[idx]['label']
        if self.transform:
            image = self.transform(image)
        return image, label

train_ds = FoodDataset(train_ds, transform=transform)
test_ds = FoodDataset(val_ds, transform=transform)

In [ ]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=True)

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, padding=2)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(16 * 32 * 32, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 101)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = CNN()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# Training loop
max_epochs = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

it_history = []
elapsed_time = time.time()
for epoch in range(max_epochs):
   epoch_loss = 0.0
   for ibatch, data in enumerate(train_loader, 0):
      batch_loss = 0.0

      # get the inputs; data is a list of [inputs, labels]
      inputs, labels = data

      # zero the parameter gradients
      optimizer.zero_grad()

      # forward + backward + optimize
      outputs = model(inputs)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

      batch_loss += loss.item()
      if (ibatch % 2000 == 1999):
         print(f'Epoch [{epoch}], Batch [{ibatch}], Batch Loss: {batch_loss:.7f}')

   epoch_loss += batch_loss
   it_history.append([epoch, epoch_loss])

elapsed_time = time.time() - elapsed_time
print(f'Elapsed Time is {elapsed_time} seconds')

/opt/anaconda3/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:890: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Elapsed Time is 2713.7418508529663 seconds


In [ ]:
PATH = './food_cnn_1.pth'
torch.save(model.state_dict(), PATH)

In [ ]:
# Training confusion matrix and accuracy
predicted_list = []
labels_list = []
with torch.no_grad():
   for images, labels in train_loader:
      images, labels = images.to(device), labels.to(device)
      outputs = model(images)
      _, predicted = torch.max(outputs.data, 1)
      predicted_list.extend(predicted.tolist())
      labels_list.extend(labels.tolist())

confusion_matrix_train = pd.crosstab(labels_list, predicted_list)
# confusion_matrix_train.columns = classes
# confusion_matrix_train.index = classes

n_total = len(labels_list)
n_correct = np.sum([1 if labels_list[i] == predicted_list[i] else 0 for i in range(n_total)])

accuracy = n_correct / n_total
print(f'Training Accuracy = {accuracy:.7f}')

Training Accuracy = 0.0910363


In [ ]:
model = CNN()
model.to(device)
model.load_state_dict(torch.load(PATH, weights_only=True))

<All keys matched successfully>

In [ ]:
# Testing confusion matrix and accuracy
predicted_list = []
labels_list = []
with torch.no_grad():
   for images, labels in test_loader:
      images, labels = images.to(device), labels.to(device)
      outputs = model(images)
      _, predicted = torch.max(outputs.data, 1)
      predicted_list.extend(predicted.tolist())
      labels_list.extend(labels.tolist())

confusion_matrix_test = pd.crosstab(labels_list, predicted_list)
# confusion_matrix_test.columns = classes
# confusion_matrix_test.index = classes

n_total = len(labels_list)
n_correct = np.sum([1 if labels_list[i] == predicted_list[i] else 0 for i in range(n_total)])

accuracy = n_correct / n_total
print(f'Testing Accuracy = {accuracy:.7f}')

/opt/anaconda3/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:890: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Testing Accuracy = 0.0921980
